In [ ]:
from gurobipy import Model, GRB, quicksum
import pandas as pd
import scipy.stats as stats
from scipy.special import gamma, gammaincc
import numpy as np

# -------------- data --------------
new_df = pd.read_csv('/Users/wenbo.wu/Documents/Documents - Wenbo WU/Thesis/RTE_use_case/RTE_dashboard/RTE/new_df_AlphaCorrected.csv', sep=',')
N = 224
alpha_init = new_df['alpha'].head(N).tolist()
V = new_df['Volume'].head(N).tolist()
K = [146 * V[i] / (8.314 * 293.15 * (10 ** 3)) for i in range(N)]
T_years = 17
Y = 365
Delta_P = [round(float(new_df.iloc[i]['Pression Nominale Relative'] - new_df['CR_threshold'].iloc[i]), 3) 
           for i in range(N)]
Age_2023 = new_df['Age 2023'].head(N).tolist()

gamma_alpha = 5.682011911867158 
gamma_beta  = 0.25018840842979584

C_COPER = 15600
C_MANT  = 48000

mu_COPER = 1.6
mu_MANT  = 4.2
M_cr = 1000

first_year_leaks = [Y * alpha_init[i] * K[i] for i in range(N)]
cr_2023 = new_df['cr'].head(N).tolist()


def compute_gamma_h(age, gamma_alpha, gamma_beta):
    z = gamma_beta * age
    if z <= 0:
        return 0.0
    num = z ** (gamma_alpha - 1) * gamma_beta * np.exp(-z)
    den = gamma(gamma_alpha) * gammaincc(gamma_alpha, z)
    return num / den


# =============== 用固定 C_leak 的函数建模 =================
def solve_model_with_C_leak(C_leak):
    mdl = Model("leak_decision_final")
    mdl.Params.OutputFlag = 1
    mdl.Params.MIPGap = 0.05
    mdl.Params.TimeLimit = 300
    mdl.Params.IntFeasTol = 1e-6
    mdl.Params.NumericFocus = 2

    # -------------- decision variables --------------
    x_COPER = mdl.addVars(N, T_years, vtype=GRB.BINARY, name="x_COPER")
    x_MANT  = mdl.addVars(N, T_years, vtype=GRB.BINARY, name="x_MANT")

    last_action = mdl.addVars(N, T_years+1, vtype=GRB.BINARY, name="last_action")
    cum_leak = mdl.addVars(N, T_years, lb=0, name="cum_leak")
    yearly_leak = mdl.addVars(N, T_years, lb=0, name="yearly_leak")
    CR = mdl.addVars(N, T_years+1, lb=0, name="CR")

    # initialization
    for i in range(N):
        mdl.addConstr(last_action[i,0] == 0)
        h_2024 = compute_gamma_h(Age_2023[i] + 1, gamma_alpha, gamma_beta)
        mdl.addConstr(CR[i,0] == cr_2023[i] + h_2024)

    # CR 演化
    for i in range(N):
        for t in range(1, T_years+1):
            no_action = 1 - (x_COPER[i,t-1] + x_MANT[i,t-1])
            h_cr = compute_gamma_h(Age_2023[i] + t, gamma_alpha, gamma_beta)
            mdl.addConstr(CR[i,t] >= CR[i,t-1] + h_cr - M_cr * (1 - no_action))
            mdl.addConstr(CR[i,t] <= CR[i,t-1] + h_cr + M_cr * (1 - no_action))

            coper_target = CR[i,t-1] / mu_COPER
            mdl.addConstr(CR[i,t] >= coper_target - M_cr * (1 - x_COPER[i,t-1]))
            mdl.addConstr(CR[i,t] <= coper_target + M_cr * (1 - x_COPER[i,t-1]))

            mant_target = CR[i,t-1] / mu_MANT
            mdl.addConstr(CR[i,t] >= mant_target - M_cr * (1 - x_MANT[i,t-1]))
            mdl.addConstr(CR[i,t] <= mant_target + M_cr * (1 - x_MANT[i,t-1]))

    # last_action update
    M_time = T_years + 5
    for i in range(N):
        for t in range(1, T_years+1):
            action_t1 = x_COPER[i,t-1] + x_MANT[i,t-1]
            mdl.addConstr(last_action[i,t] >= (t-1) - M_time * (1 - action_t1))
            mdl.addConstr(last_action[i,t] <= (t-1) + M_time * (1 - action_t1))
            mdl.addConstr(last_action[i,t] >= last_action[i,t-1] - M_time * action_t1)
            mdl.addConstr(last_action[i,t] <= last_action[i,t-1] + M_time * action_t1)

    # yearly leak calculation
    BIG_M_LEAK = 1e6
    for i in range(N):
        for t in range(T_years):
            mdl.addConstr(yearly_leak[i,t] == (CR[i,t] * Delta_P[i] * (10**5)) * K[i])

    # cumulative leak logic
    LEAK_TH = 2.0
    for i in range(N):
        # year 0: cum_leak = yearly_leak
        mdl.addConstr(cum_leak[i,0] == yearly_leak[i,0])

        for t in range(1, T_years):
            no_action_prev = 1 - (x_COPER[i,t-1] + x_MANT[i,t-1])
            mdl.addConstr(cum_leak[i,t] >= cum_leak[i,t-1] + yearly_leak[i,t] - BIG_M_LEAK * (1 - no_action_prev))
            mdl.addConstr(cum_leak[i,t] <= cum_leak[i,t-1] + yearly_leak[i,t] + BIG_M_LEAK * (1 - no_action_prev))

            action_prev = x_COPER[i,t-1] + x_MANT[i,t-1]
            mdl.addConstr(cum_leak[i,t] >= yearly_leak[i,t] - BIG_M_LEAK * (1 - action_prev))
            mdl.addConstr(cum_leak[i,t] <= yearly_leak[i,t] + BIG_M_LEAK * (1 - action_prev))

        for t in range(T_years):
            x_this_year = x_COPER[i,t] + x_MANT[i,t]
            mdl.addConstr(x_this_year <= 1)
            mdl.addConstr(cum_leak[i,t] >= LEAK_TH - BIG_M_LEAK * (1 - x_this_year))

    # objective: 使用固定 C_leak
    leak_cost = quicksum(
        (CR[i,t] * Delta_P[i] * (10**5)) * K[i] * C_leak
        for i in range(N) for t in range(T_years)
    )
    action_cost = quicksum(
        C_COPER * x_COPER[i,t] + C_MANT * x_MANT[i,t]
        for i in range(N) for t in range(T_years)
    )
    mdl.setObjective(action_cost + leak_cost, GRB.MINIMIZE)

    mdl.optimize()

    if mdl.status == GRB.OPTIMAL or mdl.status == GRB.TIME_LIMIT:
        # output results
        print(f"\n✅ C_leak = {C_leak:,.0f} | Total Cost: {mdl.objVal:,.0f}")
        
        print("\n=== Detailed Maintenance Plan and Cumulative Leak Verification ===")
        total_leak_cost = 0
        total_action_cost = 0

        for i in range(N):
            act_type = "No action"
            print(f"\n{'='*70}")
            print(f"CSEM {i+1} (Initial CR={cr_2023[i]:.1f}, Volume={V[i]:.1f}m³, csemCur = {new_df['csemCur'].iloc[i]:.1f}, Age2023={Age_2023[i]:.1f} years), K ={K[i]:.6f}")
            print(f"{'='*70}")

            csem_leak_cost = 0
            csem_action_cost = 0

            for t in range(T_years):
                if t == 0:
                    cr_start = cr_2023[i] 
                    cr_end = CR[i,0].X
                    leak_kg = yearly_leak[i,t].X 
                    leak_cost_t = leak_kg * C_leak
                    csem_leak_cost += leak_cost_t
                else:
                    cr_start = CR[i,t-1].X
                    cr_end = CR[i,t].X
                    leak_kg = yearly_leak[i,t].X 
                    leak_cost_t = leak_kg * C_leak
                    csem_leak_cost += leak_cost_t

                    act_type = ""
                    act_cost = 0
                    if x_COPER[i,t-1].X > 0.5:
                        act_type = "COPER"
                        act_cost = C_COPER
                        csem_action_cost += act_cost
                    elif x_MANT[i,t-1].X > 0.5:
                        act_type = "MANT" 
                        act_cost = C_MANT
                        csem_action_cost += act_cost
                    else:
                        act_type = "No action"

                    total_leak_cost += leak_cost_t
                    if act_cost > 0:
                        total_action_cost += act_cost

                cr_reduction = (cr_start - cr_end) / cr_start * 100 if cr_start > 0 else 0
                cum_leak_val = cum_leak[i,t].X
                valid_check = "❌" if (cum_leak_val >= 2) else "✅"

                print(f"  Y{t+1:2d}: {act_type:<8} | "
                      f"CR: {cr_start:.3f}→{cr_end:.3f} "
                      f"(-{cr_reduction:5.1f}%) | "
                      f"Leak{leak_kg:>6.3f}kg | "
                      f"Cum_Leak{cum_leak_val:>6.3f}kg | {valid_check}")

            print(f"  CSEM{i+1} Action cost{csem_action_cost:>9,.0f} + Leak cost{csem_leak_cost:>10,.0f} = Total {csem_action_cost+csem_leak_cost:>11,.0f}")
            print(f"  Total leak: {sum(yearly_leak[i,t].X for t in range(T_years)):.1f} kg")
            print(f"  Total COPER action: {sum(x_COPER[i,t].X for t in range(T_years)):.0f}")
            print(f"  Total MANT action: {sum(x_MANT[i,t].X for t in range(T_years)):.0f}")

        print(f"\n{'='*70}")
        print(f"🎯 Total: action cost {total_action_cost:>12,.0f} + leak cost {model_leak_cost:>12,.0f} = total cost {mdl.objVal:>12,.0f}")
        print(f"{'='*70}")
        print(f"Total leak over entire horizon: {sum(yearly_leak[i,t].X for i in range(N) for t in range(T_years)):.1f} kg")
        print(f"Total COPER actions: {sum(x_COPER[i,t].X for i in range(N) for t in range(T_years)):.0f}")
        print(f"Total MANT actions: {sum(x_MANT[i,t].X for i in range(N) for t in range(T_years)):.0f}")
    
        total_coper_actions = sum(x_COPER[i,t].X for i in range(N) for t in range(T_years))
        total_mant_actions = sum(x_MANT[i,t].X for i in range(N) for t in range(T_years))
        total_leak_kg = sum(yearly_leak[i,t].X for i in range(N) for t in range(T_years))
        model_leak_cost = mdl.objVal - total_action_cost

        return {
            "C_leak": C_leak,
            "total_cost": mdl.objVal,
            "total_leak_kg": total_leak_kg,              # 新增
            "leak_cost": total_leak_cost,
            "action_cost": total_action_cost,
            "total_coper_actions": total_coper_actions,  # 新增
            "total_mant_actions": total_mant_actions,    # 新增
            "model_leak_cost": model_leak_cost,          # 新增
            "status": mdl.status
        }
    else:
        print(f"❌ Optimization status with C_leak={C_leak}: {mdl.status}")
        return {
            "C_leak": C_leak,
            "total_cost": None,
            "leak_cost": None,
            "action_cost": None,
            "total_coper_actions": None,
            "total_mant_actions": None
            # "status": mdl.status
        }


# =============== 参数扫描（敏感性分析） ================
base_C_leak = 25200 * 0.100   # 你原来基准值的一部分
C_leak_list = [
    base_C_leak * 0.01,
    base_C_leak * 0.1,
    base_C_leak * 0.5,
    base_C_leak * 1,
    base_C_leak * 1.5,
    base_C_leak * 2.0,
    base_C_leak * 3.0,
    base_C_leak * 4.0,
    base_C_leak * 5.0
]

# 也可以改成固定数值列表，比如：
# C_leak_list = [15000, 20000, 25000, 30000, 35000]

results = []
for C_leak in C_leak_list:
    res = solve_model_with_C_leak(C_leak)
    results.append(res)
pd.set_option("display.float_format", "{:,.0f}".format)
# 保存结果（可选）
result_df = pd.DataFrame(results)
print("\n========== Summary of Sensitivity Analysis ==========")
print(result_df)
result_df.to_csv("sensitivity_analysis_C_leak.csv", index=False)

# 替换你原来的 print：
print('number of coper actions for each C_LEAK:')
for res in results:
    print(f"  C_leak={res['C_leak']:,.3f}: {res['total_coper_actions']:.0f} COPER actions") 

print('number of mant actions for each C_LEAK:')
for res in results:
    print(f"  C_leak={res['C_leak']:,.3f}: {res['total_mant_actions']:.0f} MANT actions")

Set parameter MIPGap to value 0.05
Set parameter TimeLimit to value 300
Set parameter IntFeasTol to value 1e-06
Set parameter NumericFocus to value 2
Gurobi Optimizer version 10.0.1 build v10.0.1rc0 (mac64[arm])

CPU model: Apple M4 Pro
Thread count: 12 physical cores, 12 logical processors, using up to 12 threads

Optimize a model with 64512 rows, 23296 columns and 221536 nonzeros
Model fingerprint: 0x1291897c
Variable types: 11648 continuous, 11648 integer (11648 binary)
Coefficient statistics:
  Matrix range     [2e-01, 1e+06]
  Objective range  [5e+00, 5e+04]
  Bounds range     [1e+00, 1e+00]
  RHS range        [3e-02, 1e+06]
Presolve removed 64498 rows and 23289 columns
Presolve time: 0.06s
Presolved: 14 rows, 7 columns, 60 nonzeros
Variable types: 3 continuous, 4 integer (4 binary)
Found heuristic solution: objective 564628.20322

Explored 0 nodes (0 simplex iterations) in 0.08 seconds (0.19 work units)
Thread count was 12 (of 12 available processors)

Solution count 1: 564628 

